# DUSC Data Exploration

## Import relevant libraries

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from eda_toolkit import generate_table1

## Paths

In [ ]:
raw_data_path = "../data/raw/"
processed_data_path = "../data/processed/"

In [ ]:
df = pd.read_parquet("../data/processed/df_eda.parquet")

In [ ]:
df["dramatic"].value_counts()

In [ ]:
df["dramatic"].value_counts(1)

In [ ]:
df["shape"].value_counts()

## Correlation between all variables in model

In [ ]:
from eda_toolkit import flex_corr_matrix

flex_corr_matrix(
    df=df,
    cols=df.select_dtypes(include=np.number).columns.to_list(),
    annot=True,
    cmap="coolwarm",
    figsize=(10, 8),
    title="US Census Correlation Matrix",
    xlabel_alignment="right",
    label_fontsize=8,
    tick_fontsize=8,
    xlabel_rot=45,
    ylabel_rot=0,
    text_wrap=50,
    vmin=-1,
    vmax=1,
    cbar_label="Correlation Index",
    triangular=True,
)

In [ ]:
df.columns.to_list()

In [ ]:
df["State"].value_counts()

In [ ]:
df["cluster_status"] = np.select(
    [df["Country"] != "USA", df["latitude"].isna(), df["cluster_id"].notna()],
    ["Not assessed (non-US)", "Not assessed (no coordinates)", "In a cluster"],
    default="Assessed, isolated",
)

In [ ]:
df["cluster_status"].value_counts()

In [ ]:
not_usa = df[df["country"] != "USA"]

In [ ]:
df[(df["cluster_id"].isna()) & (df["Country"] == "USA")]

In [ ]:
# X = pd.read_parquet("../data/processed/X.parquet")
# print(X[["days_since_uap_event", "occurred_year"]].corr(method="spearman"))
# print(X.groupby("occurred_year")["days_since_uap_event"].agg(["min", "max"]))

In [ ]:
us_null = df[(df["Country"] == "USA") & df["cluster_id"].isna()]
print(us_null["latitude"].isna().value_counts())
# True  -> no coordinates, never eligible
# False -> geocoded, DBSCAN called it noise

In [ ]:
usa_only = df[df["country"] == "USA"]

In [ ]:
pd.crosstab(usa_only["State"], df["shape"])

## Claim 1 and 2: flag rate and volume, per year

In [ ]:
print(
    df.groupby("occurred_year")["dramatic"]
    .agg(n="size", flagged="sum", rate="mean")
    .assign(rate=lambda d: (d["rate"] * 100).round(1))
)

# Claim 3: is 2022 evenly spread across months, or truncated?
print(
    df[df["occurred_year"] == 2022]
    .groupby("occurred_month")["dramatic"]
    .agg(n="size", rate="mean")
)

## Table 1

In [ ]:
df.columns.to_list()

In [ ]:
TABLE1_COLS = [
    "occurred_year",
    "occurred_month",
    "Shape",
    "Country",
    "State",
    "is_night",
    "is_weekend",
    "has_media",
    "in_cluster",
    "text_is_summary_only",
    # "exp_drone",
    # "exp_rocket",
    # "exp_balloon",
    # "exp_aircraft",
    # "exp_starlink",
    # "exp_lantern",
    # "exp_satellite",
    # "exp_certain",
    "Explanation",
    "dramatic",
]

tbl = generate_table1(
    df[TABLE1_COLS].copy(),
    value_counts=True,
    groupby_col="dramatic",
    drop_columns=["Type", "Mean", "SD", "Median", "Min", "Max", "Mode"],
    apply_bonferroni=True,
)

In [ ]:
tbl

In [ ]:
tbl.to_csv("../data/processed/df_table1.csv")

# EDA

In [ ]:
# World Map: Heatmap KDE Density Plot

# Define File Paths for Files inside /geo_maps/world
geo_maps_dir_world = Path("../geo_maps/world")
matches = list(geo_maps_dir_world.rglob("ne_110m_admin_0_countries.shp"))

if not matches:
    # Fallback search for any .shp file if specific filename isn't matched
    matches = list(geo_maps_dir_world.rglob("*.shp"))

if matches:
    world_path = str(matches[0])
    print(f"Loading shapefile from: {world_path}")
    world = gpd.read_file(world_path)
else:
    raise FileNotFoundError(f"No .shp files found in {geo_maps_dir_world}")

# Build sighting GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lonn
)

# If there's no access to a .prj file: explicitly assign CRS (EPSG:4326)
world = world.set_crs("EPSG:4326", allow_override=True)

# Plot the KDE Heatmap (Sightings Gradient)
fig, ax = plt.subplots(figsize=(10, 8))

world.plot(ax=ax, color="#e0e0e0", edgecolor="white", linewidth=0.5)

# Overlay the KDE based on sighting positions
sns.kdeplot(
    x=gdf.geometry.x,
    y=gdf.geometry.y,
    cmap="YlOrRd",  # Yellow-Orange-Red gradient color scheme
    fill=True,
    alpha=0.6,
    levels=15,  # Smoothness/resolution of density contours
    ax=ax,
)

# Plot individual sighting points on top
gdf.plot(
    ax=ax,
    color="black",
    markersize=8,
    alpha=0.7,
    label="Individual Sightings",
)

# Framing & Formatting
minx, miny, maxx, maxy = gdf.total_bounds
buffer = 1.0  # Buffer in degrees
ax.set_xlim(minx - buffer, maxx + buffer)
ax.set_ylim(miny - buffer, maxy + buffer)

ax.set_title(
    f"Sightings Heatmap (Total Reports: {len(gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# US Map 1: Heatmap KDE Density Plot

# Define File Paths for Files inside /geo_maps/us
us_dir = Path("../geo_maps/us")

# Paths to the US State base map & US State shapefiles
us_states_path = us_dir / "tl_2023_us_state.shp"

# Load both shapefile layers
us_states = gpd.read_file(us_states_path)

# Build Sighting GeoDataFrame & Align CRS
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lon
)

# Align CRS for the shapefile to match sightings data
us_states = us_states.to_crs(gdf.crs)

# Filter sighting data down to Continental US (CONUS)
us_gdf = gdf.cx[-125:-66, 24:50]

# Build Layered US Map
plt.close("all")  # Prevent plot memory overlap
fig, ax = plt.subplots(figsize=(14, 10))

# Layer 1: US State Boundaries Base Map (Light gray fill w/ white state borders)
us_states.plot(ax=ax, color="#e0e0e0", edgecolor="white", linewidth=0.8)


# Layer 2: KDE Heatmap (Sightings Gradient)
sns.kdeplot(
    x=us_gdf.geometry.x,
    y=us_gdf.geometry.y,
    cmap="YlOrRd",  # Yellow-Orange-Red gradient
    fill=True,
    alpha=0.6,
    levels=15,
    ax=ax,
)

# Layer 4: Individual Sighting Points
us_gdf.plot(
    ax=ax,
    color="black",
    markersize=8,
    alpha=0.7,
    label="Individual Sightings",
)

# Zoom to Continental US (CONUS)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

ax.set_title(
    f"US Sightings Heatmap (Total CONUS Reports: {len(us_gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# US Map 2: Chloropleth & Sighting Density Plot

# Define File Paths for Files inside /geo_maps/us
us_dir = Path("../geo_maps/us")
us_states_path = us_dir / "tl_2023_us_state.shp"

us_states = gpd.read_file(us_states_path)

# Build Sighting GeoDataFrame & Align CRS
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lon
)

# Align CRS
us_states = us_states.to_crs(gdf.crs)

# Filter sighting data down to Continental US (CONUS)
us_gdf = gdf.cx[-125:-66, 24:50]

# Spatial Join & Aggregate Counts per State
# Join points into state polygons
joined = gpd.sjoin(us_states, us_gdf, how="left", predicate="contains")

# Group by state identifier (STUSPS is two-letter abbreviation, or rely on GEOID)
sighting_counts = (
    joined.groupby("STUSPS").size().reset_index(name="sighting_count")
)

# Merge counts back to original state polygons
us_states = us_states.merge(sighting_counts, on="STUSPS", how="left")
us_states["sighting_count"] = us_states["sighting_count"].fillna(0)

# Build Choropleth Map
plt.close("all")
fig, ax = plt.subplots(figsize=(14, 10))

# Layer 1: Choropleth filled by state sighting counts
us_states.plot(
    column="sighting_count",
    cmap="YlOrRd",  # Yellow-Orange-Red gradient
    linewidth=0.8,
    ax=ax,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Total Sightings per State",
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.02,
    },
)

# Layer 2: Individual Sighting Points (Optional: lower alpha for clarity)
us_gdf.plot(
    ax=ax,
    color="black",
    markersize=4,
    alpha=0.4,
    label="Individual Sightings",
)

# Framing & Formatting (CONUS Zoom)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

ax.set_title(
    f"US State Choropleth & Sighting Density (Total CONUS Reports: {len(us_gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()